# Interpretability Analysis

Two complementary methods:
1. **Grad-CAM** — which spectral positions drive predictions (CNN / Hybrid)
2. **Attention maps** — which patches the model attends to (Transformer / Hybrid)

Research goal: do highlighted regions correspond to known spectral features?

In [ ]:
import sys; sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from src.utils.config import load_config
from src.utils.checkpoint import load_best_model
from src.data.registry import DataRegistry
from src.data.preprocessing import SpectralPreprocessor
from src.data.dataset import SpectralDataset
from src.models.registry import get_model
from src.interpretability.gradcam1d import GradCAM1D
from src.interpretability.attention_viz import (
    extract_attention, cls_to_patch_attention,
    attention_rollout, patch_attention_to_signal, per_class_attention
)

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

SHARED_CLASSES = [0, 2, 3, 5, 6]
CLASS_COLORS   = plt.cm.tab10(np.linspace(0, 1, len(SHARED_CLASSES)))
SIGNAL_LENGTH  = 1000
PATCH_SIZE     = 20

In [ ]:
# ================================================================
# LOAD DATA AND MODELS
# ================================================================
cfg = load_config(
    '../configs/data/splits.yaml',
    '../configs/data/preprocessing.yaml',
)

registry = DataRegistry(data_root='../data/raw', cfg=cfg)
registry.load_all()

X_ref, y_ref = registry.get_arrays('reference')
preprocessor = SpectralPreprocessor.from_config(cfg['preprocessing'])
preprocessor.fit(X_ref)

X_clin, y_clin = registry.get_arrays('2018clinical')
X_clin_proc = preprocessor.transform(X_clin)

# Build test dataset (shared classes only for clinical analysis)
X_ref_proc = preprocessor.transform(X_ref)
ref_dataset  = SpectralDataset(X_ref_proc,  y_ref,  class_filter=SHARED_CLASSES)
clin_dataset = SpectralDataset(X_clin_proc, y_clin, class_filter=SHARED_CLASSES)

# --- Load models (update paths after training) ---
EXPERIMENT_DIRS = {
    'cnn':         '../experiments/cnn_best',
    'hybrid':      '../experiments/hybrid_best',
    'transformer': '../experiments/transformer_best',
}

models = {}
for name, exp_dir in EXPERIMENT_DIRS.items():
    model_cfg = load_config(f'{exp_dir}/config.yaml')
    m = get_model(name, dict(model_cfg))
    load_best_model(exp_dir, m)
    m.eval()
    models[name] = m
    print(f'Loaded {name}')

---
## Part 1 — Grad-CAM: which spectral positions drive CNN predictions?

In [ ]:
# ================================================================
# GRAD-CAM: Mean saliency map per class (shared classes)
# ================================================================
gcam = GradCAM1D(models['cnn'])
N_SAMPLES_PER_CLASS = 30

fig, axes = plt.subplots(len(SHARED_CLASSES), 1, figsize=(14, 4 * len(SHARED_CLASSES)))

for ax, cls, color in zip(axes, SHARED_CLASSES, CLASS_COLORS):
    # Collect samples for this class
    idx = [i for i in range(len(ref_dataset)) if ref_dataset[i][1].item() == cls][:N_SAMPLES_PER_CLASS]
    
    cams, signals = [], []
    for i in idx:
        x, _ = ref_dataset[i]
        cam = gcam.compute(x.unsqueeze(0), target_class=cls, signal_length=SIGNAL_LENGTH)
        cams.append(cam)
        signals.append(x.squeeze().numpy())
    
    mean_cam    = np.mean(cams, axis=0)
    mean_signal = np.mean(signals, axis=0)
    positions   = np.arange(SIGNAL_LENGTH)

    # Plot signal
    ax.plot(positions, mean_signal, color=color, lw=1.5, alpha=0.8, label=f'Mean signal (class {cls})')
    
    # Overlay Grad-CAM as shaded region
    ax2 = ax.twinx()
    ax2.fill_between(positions, mean_cam, alpha=0.35, color='orangered', label='Grad-CAM saliency')
    ax2.set_ylim(0, 1.5)
    ax2.set_ylabel('Saliency', fontsize=9)
    ax2.tick_params(axis='y', labelsize=8)
    
    ax.set_title(f'Class {cls} — CNN Grad-CAM (mean of {len(idx)} samples)', fontweight='bold')
    ax.set_xlabel('Spectral position')
    ax.set_ylabel('Normalised intensity')
    ax.legend(loc='upper left', fontsize=9)

plt.suptitle('CNN Grad-CAM — which positions drive each class prediction?', 
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../experiments/interpretability_gradcam_per_class.png', 
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ================================================================
# GRAD-CAM: Source vs clinical — same class, same model
# Do saliency regions shift under domain shift?
# ================================================================
gcam = GradCAM1D(models['cnn'])
target_cls = 0   # Pick one shared class

def mean_cam_for_class(dataset, cls, n=20):
    idx = [i for i in range(len(dataset)) if dataset[i][1].item() == cls][:n]
    cams = [gcam.compute(dataset[i][0].unsqueeze(0), target_class=cls) for i in idx]
    signals = [dataset[i][0].squeeze().numpy() for i in idx]
    return np.mean(cams, axis=0), np.mean(signals, axis=0)

ref_cam,  ref_sig  = mean_cam_for_class(ref_dataset,  target_cls)
clin_cam, clin_sig = mean_cam_for_class(clin_dataset, target_cls)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
positions = np.arange(SIGNAL_LENGTH)

for ax, sig, cam, label, color in [
    (axes[0], ref_sig,  ref_cam,  'Reference (source)', '#1D9E75'),
    (axes[1], clin_sig, clin_cam, '2018clinical (OOD)',  '#D85A30'),
]:
    ax.plot(positions, sig, color=color, lw=1.5)
    ax2 = ax.twinx()
    ax2.fill_between(positions, cam, alpha=0.4, color='orangered')
    ax2.set_ylim(0, 1.5)
    ax.set_title(f'Class {target_cls} — {label}', fontweight='bold')
    ax.set_ylabel('Intensity')

axes[1].set_xlabel('Spectral position')
plt.suptitle(f'Grad-CAM: same class ({target_cls}), same model, source vs clinical\n'
             'Consistent saliency regions = model uses same features in both domains',
             fontsize=12)
plt.tight_layout()
plt.savefig('../experiments/interpretability_gradcam_domain_shift.png',
            dpi=150, bbox_inches='tight')
plt.show()

---
## Part 2 — Attention maps: what does the Transformer look at?

In [ ]:
# ================================================================
# ATTENTION: Per-class mean CLS→patch attention
# ================================================================
class_attn = per_class_attention(
    models['transformer'],
    ref_dataset,
    class_ids=SHARED_CLASSES,
    n_samples=30,
    patch_size=PATCH_SIZE,
    signal_length=SIGNAL_LENGTH,
)

fig, axes = plt.subplots(len(SHARED_CLASSES), 1, figsize=(14, 4 * len(SHARED_CLASSES)))

for ax, cls, color in zip(axes, SHARED_CLASSES, CLASS_COLORS):
    # Mean signal
    idx = [i for i in range(len(ref_dataset)) if ref_dataset[i][1].item() == cls][:30]
    mean_sig = np.mean([ref_dataset[i][0].squeeze().numpy() for i in idx], axis=0)
    
    ax.plot(np.arange(SIGNAL_LENGTH), mean_sig, color=color, lw=1.5)
    ax2 = ax.twinx()
    ax2.fill_between(np.arange(SIGNAL_LENGTH), class_attn[cls],
                     alpha=0.35, color='steelblue', label='Attention')
    ax2.set_ylim(0, 1.5)
    ax2.set_ylabel('Attention weight', fontsize=9)
    ax.set_title(f'Class {cls} — Transformer CLS attention', fontweight='bold')
    ax.set_xlabel('Spectral position')
    ax.set_ylabel('Normalised intensity')

plt.suptitle('Transformer CLS→patch attention — which spectral regions are attended to?',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../experiments/interpretability_attention_per_class.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ================================================================
# ATTENTION ROLLOUT: Multi-layer attribution
# ================================================================
# Pick one sample per class and compute rollout vs single-layer attention

fig, axes = plt.subplots(len(SHARED_CLASSES), 1, figsize=(14, 4 * len(SHARED_CLASSES)))

for ax, cls, color in zip(axes, SHARED_CLASSES, CLASS_COLORS):
    idx = next(i for i in range(len(ref_dataset)) if ref_dataset[i][1].item() == cls)
    x, _ = ref_dataset[idx]
    x_t = x.unsqueeze(0)
    
    attn_maps = extract_attention(models['transformer'], x_t)
    
    # Single last-layer attention
    last_attn = cls_to_patch_attention(attn_maps, layer=-1, head='mean')
    last_attn_sig = np.repeat(last_attn, PATCH_SIZE)[:SIGNAL_LENGTH]
    
    # Rollout
    rollout = attention_rollout(attn_maps)
    rollout_sig = np.repeat(rollout, PATCH_SIZE)[:SIGNAL_LENGTH]
    
    sig = x.squeeze().numpy()
    pos = np.arange(SIGNAL_LENGTH)
    
    ax.plot(pos, sig, color=color, lw=1.2, alpha=0.7, label='Signal')
    ax2 = ax.twinx()
    ax2.plot(pos, last_attn_sig, color='steelblue',  lw=1.5, alpha=0.7, label='Last layer')
    ax2.plot(pos, rollout_sig,   color='darkviolet', lw=1.5, alpha=0.7, linestyle='--', label='Rollout')
    ax2.set_ylim(0, 1.5)
    ax2.legend(fontsize=8)
    ax.set_title(f'Class {cls} — last-layer attention vs rollout', fontweight='bold')
    ax.set_xlabel('Spectral position')
    ax.set_ylabel('Intensity')

plt.suptitle('Attention: last layer vs rollout (multi-layer attribution)', 
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../experiments/interpretability_attention_rollout.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ================================================================
# COMPARISON: Grad-CAM (CNN) vs Attention (Transformer) for same sample
# Shows whether both methods agree on which regions matter
# ================================================================
gcam_cnn    = GradCAM1D(models['cnn'])
gcam_hybrid = GradCAM1D(models['hybrid'])

target_cls = SHARED_CLASSES[0]
idx = next(i for i in range(len(ref_dataset)) if ref_dataset[i][1].item() == target_cls)
x, _ = ref_dataset[idx]
x_t  = x.unsqueeze(0)
sig  = x.squeeze().numpy()
pos  = np.arange(SIGNAL_LENGTH)

cam_cnn    = gcam_cnn.compute(x_t,    target_class=target_cls)
cam_hybrid = gcam_hybrid.compute(x_t, target_class=target_cls)
attn_maps  = extract_attention(models['transformer'], x_t)
attn_sig   = np.repeat(cls_to_patch_attention(attn_maps, -1, 'mean'), PATCH_SIZE)[:SIGNAL_LENGTH]

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

for ax, saliency, label, color in [
    (axes[0], cam_cnn,    'CNN Grad-CAM',        '#D85A30'),
    (axes[1], cam_hybrid, 'Hybrid Grad-CAM',     '#534AB7'),
    (axes[2], attn_sig,   'Transformer attention', '#0F6E56'),
]:
    ax.plot(pos, sig, color='#888780', lw=1.2, alpha=0.7)
    ax2 = ax.twinx()
    ax2.fill_between(pos, saliency, alpha=0.45, color=color, label=label)
    ax2.set_ylim(0, 1.5)
    ax2.legend(loc='upper right', fontsize=9)
    ax.set_ylabel('Intensity')

axes[2].set_xlabel('Spectral position')
plt.suptitle(f'Method comparison — class {target_cls}, same sample\n'
             'Agreement between methods = robust attribution',
             fontsize=12)
plt.tight_layout()
plt.savefig('../experiments/interpretability_method_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()